In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'reddit_mental_health')

import os, random
import math
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

import matplotlib.pyplot as plt


In [ ]:
SEED = 42
DATA_CSV = _paths.input_path('filtered_dataset.csv')
TEXT_COL = "post"
LABEL_COL = "label"

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256


PER_DEVICE_TRAIN_BATCH = 32
PER_DEVICE_EVAL_BATCH = 32
GRADIENT_ACCUMULATION = 1

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

OUTPUT_DIR = _paths.output_subdir('distilbert_optimized_runs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
df = pd.read_csv(_paths.input_path(DATA_CSV))
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
print("Label distribution:\n", df[LABEL_COL].value_counts())


le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df[LABEL_COL].astype(str))
num_labels = len(le.classes_)
class_names = list(le.classes_)
print("num_labels:", num_labels)
print("class_names:", class_names)


In [ ]:
idx = df.index.values
temp_idx, test_idx = train_test_split(
    idx, test_size=0.20, stratify=df["label_encoded"], random_state=SEED
)


train_idx, val_idx = train_test_split(
    temp_idx,
    test_size=0.125,
    stratify=df.loc[temp_idx, "label_encoded"],
    random_state=SEED,
)

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = df.loc[val_idx].reset_index(drop=True)
test_df = df.loc[test_idx].reset_index(drop=True)

print("train / val / test sizes:", len(train_df), len(val_df), len(test_df))


ds_train = Dataset.from_pandas(
    train_df[[TEXT_COL, "label_encoded"]].rename(
        columns={TEXT_COL: "text", "label_encoded": "labels"}
    )
)
ds_val = Dataset.from_pandas(
    val_df[[TEXT_COL, "label_encoded"]].rename(
        columns={TEXT_COL: "text", "label_encoded": "labels"}
    )
)
ds_test = Dataset.from_pandas(
    test_df[[TEXT_COL, "label_encoded"]].rename(
        columns={TEXT_COL: "text", "label_encoded": "labels"}
    )
)

full_dataset = DatasetDict({"train": ds_train, "validation": ds_val, "test": ds_test})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)


full_tok = full_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
full_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenized sizes:", {k: len(v) for k, v in full_tok.items()})


In [ ]:
train_labels = np.array(full_tok["train"]["labels"])
classes = np.arange(num_labels)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(cw, dtype=torch.float)
print("Class weights:", cw)


In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class WeightedTrainer(Trainer):
    class_weights: Optional[torch.Tensor] = None

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")

        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        if self.class_weights is not None:
            cw = self.class_weights.to(logits.device)
            loss_fct = torch.nn.CrossEntropyLoss(weight=cw)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [ ]:
cfg = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: n for i, n in enumerate(class_names)},
    label2id={n: i for i, n in enumerate(class_names)},
)


cfg.dropout = 0.3
cfg.attention_dropout = 0.3

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=cfg)

try:
    model.gradient_checkpointing_enable()
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model loaded on", device)


In [ ]:
from transformers import Trainer
import torch


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(model.device)
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "distilbert-best"),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.06,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    save_total_limit=3,
    report_to=[],
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=full_tok["train"],
    eval_dataset=full_tok["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda pred: {
        "accuracy": accuracy_score(pred.label_ids, np.argmax(pred.predictions, axis=1)),
        "precision": precision_score(
            pred.label_ids,
            np.argmax(pred.predictions, axis=1),
            average="weighted",
            zero_division=0,
        ),
        "recall": recall_score(
            pred.label_ids,
            np.argmax(pred.predictions, axis=1),
            average="weighted",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            pred.label_ids, np.argmax(pred.predictions, axis=1), average="weighted"
        ),
    },
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights,
)


print("Trainer ready. GPU:", torch.cuda.is_available())


In [ ]:
trainer.train()
trainer.save_model(training_args.output_dir)
print("Training finished. Model saved to:", training_args.output_dir)


In [ ]:
preds = trainer.predict(full_tok["test"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(cm, interpolation='nearest', cmap="Blues")
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
plt.colorbar(im, ax=ax)
ax.set_title("Normalized Confusion Matrix")
plt.tight_layout()
plt.show()


report_dict = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
f1_scores = [report_dict[c]["f1-score"] for c in class_names]

plt.figure(figsize=(12, 6))
plt.bar(class_names, f1_scores)
plt.xticks(rotation=45, ha="right")
plt.ylabel("F1-score")
plt.title("Per-class F1 Scores")
plt.tight_layout()
plt.show()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


best_model_dir = _paths.transformer_model('distilbert_optimized_runs/distilbert-best')


tokenizer = AutoTokenizer.from_pretrained(best_model_dir)
model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)


import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Best model reloaded from:", best_model_dir)


In [ ]:
def predict(text):
    inputs = tokenizer(
        text, truncation=True, padding=True, max_length=256, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_class = torch.argmax(probs, dim=-1).item()
    return pred_class, probs.cpu().numpy()


sample_text = "I am feeling very low and anxious these days."
label_id, probabilities = predict(sample_text)

print("Predicted label:", label_id)
print("Probabilities:", probabilities)


In [ ]:
from transformers import pipeline
import shap
import matplotlib.pyplot as plt
import numpy as np
from lime.lime_text import LimeTextExplainer
import torch


class_mapping = {
    0: "addiction",
    1: "adhd",
    2: "alcoholism",
    3: "anxiety",
    4: "autism",
    5: "bipolarreddit",
    6: "bpd",
    7: "depression",
    8: "healthanxiety",
    9: "lonely",
    10: "mentalhealth",
    11: "ptsd",
    12: "schizophrenia",
    13: "socialanxiety",
    14: "suicidewatch",
}


nlp_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,
)


def predict_proba(texts):
    outputs = [nlp_pipeline(t)[0] for t in texts]
    probs = []
    for out in outputs:

        sorted_out = sorted(out, key=lambda x: x['label'])
        probs.append([x['score'] for x in sorted_out])
    return np.array(probs)


explainer = shap.Explainer(nlp_pipeline)


class_names = list(class_mapping.values())
lime_explainer = LimeTextExplainer(class_names=class_names)


def explain_prediction_dl(input_text, mode="both", top_n=10):
    """
    Explain DistilBERT predictions with SHAP, LIME, or both.
    mode: "shap", "lime", or "both"
    Works with single sentence, multiple sentences, or long paragraphs.
    """
    if isinstance(input_text, str):
        texts = [input_text]
    else:
        texts = input_text

    probs = predict_proba(texts)
    for i, text in enumerate(texts):
        prob = probs[i]
        pred_idx = np.argmax(prob)
        pred_class = class_mapping[pred_idx]
        pred_score = prob[pred_idx] * 100

        print("=" * 60)
        print(f"Text {i+1}: {text}\n")
        print(f"Predicted Condition: {pred_class} ({pred_score:.2f}%)")
        print("Prediction Probabilities (top 5):")
        top5 = sorted(list(enumerate(prob)), key=lambda x: x[1], reverse=True)[:5]
        for j, p in top5:
            marker = "<-- Main Prediction" if class_mapping[j] == pred_class else ""
            print(f"  {class_mapping[j]}: {p:.3f} {marker}")

        if mode in ["lime", "both"]:
            print("\n🔹 LIME Explanation (word-level, easy to read):")
            exp = lime_explainer.explain_instance(
                text, predict_proba, num_features=top_n, labels=[pred_idx]
            )

            for word, weight in exp.as_list(label=pred_idx):
                direction = "supports" if weight > 0 else "opposes"
                print(
                    f"  - '{word}' {direction} prediction of '{pred_class}' (score={weight:.3f})"
                )

            exp.show_in_notebook(text=True)

        if mode in ["shap", "both"]:
            print("\n🔹 SHAP Explanation (token-level, deep dive):")
            shap_values = explainer([text])

            shap.plots.text(shap_values[0])

            token_scores = shap_values.values[0].sum(axis=1)
            tokens = shap_values.data[0]
            n_top = min(top_n, len(tokens))
            top_idx = np.argsort(np.abs(token_scores))[-n_top:]

            plt.figure(figsize=(8, 4))
            colors = ["green" if s > 0 else "red" for s in token_scores[top_idx]]
            plt.barh(np.array(tokens)[top_idx], token_scores[top_idx], color=colors)
            plt.xlabel("Contribution Score")
            plt.title(f"Top {n_top} tokens contributing to '{pred_class}' (SHAP)")
            plt.show()

            print("\nTop contributing tokens (SHAP textual explanation):")
            for j in top_idx[::-1]:
                direction = "supports" if token_scores[j] > 0 else "opposes"
                print(
                    f"  - '{tokens[j]}' {direction} prediction of '{pred_class}' (score={token_scores[j]:.3f})"
                )

        print(
            f"\n✅ Summary: The model predicts '{pred_class}' with {pred_score:.2f}% confidence.\n"
        )


In [ ]:
explain_prediction_dl("I am feeling very low and anxious these days.", mode="both")


In [ ]:
sample_texts = [
    "I am feeling very low and anxious these days.",
    "Today was an amazing and joyful day!",
    "Sometimes I get panic attacks in public places.",
    "I feel hopeful and positive about my future.",
]
explain_prediction_dl(sample_texts)


In [ ]:
explain_prediction_dl(
    """
Some days I feel unstoppable, energized, and overly confident, taking impulsive risks. 
Other days, I’m lethargic, hopeless, and unable to complete basic tasks. 
My mood swings unpredictably, leaving me exhausted and confused.
""",
    mode="both",
)


In [ ]:
explain_prediction_dl(
    """
I am exremely happy today and looking forward to the future!
""",
    mode="both",
)
